# Analyse du Data Drift

Objectif : détecter si la distribution des données reçues en production diffère significativement des données d'entraînement.

**Pourquoi c'est important** : si les données de production dérivent par rapport aux données d'entraînement, les prédictions du modèle deviennent moins fiables — le modèle a appris sur une distribution qui n'est plus représentative de la réalité.

On utilise **Evidently** qui calcule automatiquement des tests statistiques (KS test, PSI) pour chaque feature et génère un rapport HTML interactif.

In [2]:
import pandas as pd
import numpy as np
import json
import os
import psycopg2
from psycopg2.extras import RealDictCursor
from dotenv import load_dotenv
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.metrics import ColumnDriftMetric
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
print('✅ Imports OK')

ImportError: cannot import name 'BaseResult' from 'evidently.core' (c:\Users\axell\Documents\projets_code\credit_scoring\venv\Lib\site-packages\evidently\core\__init__.py)

## 1. Chargement des données de référence

Les données de référence sont les données d'entraînement — c'est la distribution que le modèle a apprise.
On les utilise comme baseline pour détecter si les nouvelles données s'en écartent.

In [ ]:
# Données d'entraînement = référence
ref_data = pd.read_csv('train_test/sample_train.csv')

# Features à surveiller — variables métier clés
FEATURES_TO_MONITOR = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'CNT_CHILDREN',
    'NAME_CONTRACT_TYPE',
    'DAYS_BIRTH',
    'DAYS_EMPLOYED',
]
FEATURES_TO_MONITOR = [f for f in FEATURES_TO_MONITOR if f in ref_data.columns]

print(f'Données de référence : {ref_data.shape}')
print(f'Features surveillées : {FEATURES_TO_MONITOR}')

## 2. Chargement des données de production depuis NeonDB

On récupère les prédictions loggées en production.
Chaque ligne contient les features du client qui a été scoré.

In [ ]:
def load_production_data():
    """
    Charge les données de production depuis NeonDB.
    Retourne un DataFrame avec les features des clients scorés.
    """
    conn = psycopg2.connect(os.getenv('NEON_DATABASE_URL'))
    cur = conn.cursor(cursor_factory=RealDictCursor)
    cur.execute("""
        SELECT client_id, prediction, prediction_proba, latency_ms, input_features, timestamp
        FROM predictions
        WHERE status = 'success' AND input_features IS NOT NULL
        ORDER BY timestamp DESC
        LIMIT 1000
    """)
    rows = cur.fetchall()
    cur.close()
    conn.close()

    if not rows:
        raise ValueError("Aucune donnée de production disponible")

    # Extraire les features depuis le champ JSONB
    features_list = []
    for row in rows:
        features = row['input_features'] if isinstance(row['input_features'], dict) else json.loads(row['input_features'])
        features['client_id'] = row['client_id']
        features['prediction'] = row['prediction']
        features['prediction_proba'] = row['prediction_proba']
        features['latency_ms'] = row['latency_ms']
        features_list.append(features)

    return pd.DataFrame(features_list)

prod_data = load_production_data()
print(f'Données de production : {prod_data.shape}')
print(f'Période : {prod_data.get("timestamp", ["N/A"])[0] if "timestamp" in prod_data else "N/A"}')

## 3. Rapport Evidently — Data Drift

Evidently compare la distribution de chaque feature entre les données de référence (train) et les données de production.

Pour chaque feature il calcule :
- **KS test** (Kolmogorov-Smirnov) pour les variables numériques : teste si les deux distributions sont identiques
- **PSI** (Population Stability Index) : mesure l'écart entre les distributions
- **Chi2** pour les variables catégorielles

Un drift est détecté quand le p-value du test est < 0.05 ou le PSI > 0.2.

In [ ]:
# Préparer les datasets avec uniquement les features communes
common_features = [f for f in FEATURES_TO_MONITOR if f in prod_data.columns]

ref_sample  = ref_data[common_features].dropna()
prod_sample = prod_data[common_features].dropna()

print(f'Features analysées : {common_features}')
print(f'Référence : {len(ref_sample)} lignes | Production : {len(prod_sample)} lignes')

# Rapport Evidently
report = Report(metrics=[
    DataDriftPreset(),      # drift par feature
    DataQualityPreset(),    # qualité des données (NaN, outliers...)
])

report.run(reference_data=ref_sample, current_data=prod_sample)

# Sauvegarder le rapport HTML interactif
os.makedirs('reports', exist_ok=True)
report.save_html('reports/data_drift_report.html')
print('✅ Rapport sauvegardé : reports/data_drift_report.html')

## 4. Résultats par feature

On extrait les résultats du rapport pour identifier quelles features ont drifté.

In [ ]:
# Extraire les résultats sous forme de DataFrame
report_dict = report.as_dict()
drift_results = []

for metric in report_dict['metrics']:
    if metric['metric'] == 'DataDriftTable':
        for feature, values in metric['result']['drift_by_columns'].items():
            drift_results.append({
                'feature'        : feature,
                'drift_detected' : values['drift_detected'],
                'p_value'        : round(values.get('p_value', float('nan')), 4),
                'statistic'      : round(values.get('statistic', float('nan')), 4),
                'test'           : values.get('stattest_name', 'N/A'),
            })

drift_df = pd.DataFrame(drift_results).sort_values('drift_detected', ascending=False)
print(drift_df.to_string(index=False))

n_drifted = drift_df['drift_detected'].sum()
print(f'\n{n_drifted}/{len(drift_df)} features en drift')

## 5. Drift sur les scores de prédiction

En plus du drift des features, on surveille la distribution des scores de probabilité.
Si la distribution des scores change significativement, le modèle se comporte différemment — même si les features individuelles semblent stables.

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

if 'prediction_proba' in prod_data.columns:
    prod_scores = prod_data['prediction_proba'].dropna()

    # Distribution des scores en production
    plt.figure(figsize=(10, 4))
    plt.hist(prod_scores, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    plt.axvline(x=prod_scores.mean(), color='tomato', linestyle='--', label=f'Moyenne : {prod_scores.mean():.2f}')
    plt.xlabel('Probabilité de défaut')
    plt.ylabel('Nombre de prédictions')
    plt.title('Distribution des scores de prédiction en production')
    plt.legend()
    plt.tight_layout()
    plt.savefig('reports/score_distribution.png')
    plt.show()

    print(f'Score moyen en production : {prod_scores.mean():.3f}')
    print(f'Score médian              : {prod_scores.median():.3f}')
    print(f'Taux de refus (seuil=0.49): {(prod_scores >= 0.49).mean()*100:.1f}%')

## 6. Interprétation et points de vigilance

**Comment interpréter les résultats :**

- **Aucun drift détecté** → le modèle voit des données similaires à celles sur lesquelles il a été entraîné, les prédictions sont fiables
- **Drift sur 1-2 features peu importantes** → surveillance à renforcer, pas d'action immédiate requise
- **Drift sur des features importantes** (EXT_SOURCE, AMT_CREDIT, AMT_INCOME_TOTAL) → risque de dégradation des prédictions, réentraînement du modèle à envisager
- **Drift sur les scores** → signal fort que le comportement du modèle a changé

**Limites de cette analyse :**

- Le volume de données de production est faible (simulation), ce qui peut produire des faux positifs sur les tests statistiques
- Le drift détecté ne signifie pas forcément une dégradation des performances — il faut idéalement avoir accès aux vrais labels de production pour le confirmer

**Actions recommandées si drift détecté :**

1. Analyser la cause du drift (changement de population, problème de données en amont)
2. Évaluer l'impact sur les métriques business (coût métier)
3. Envisager un réentraînement avec des données récentes si la dégradation est confirmée

In [ ]:
# Résumé final
print('=== RÉSUMÉ ANALYSE DATA DRIFT ===')
print(f'Features analysées    : {len(drift_df)}')
print(f'Features en drift     : {n_drifted}')
print(f'Taux de drift         : {n_drifted/len(drift_df)*100:.0f}%')

if n_drifted == 0:
    print('\n✅ Aucun drift détecté — modèle stable en production')
elif n_drifted <= 2:
    print('\n⚠️  Drift mineur détecté — surveillance à renforcer')
else:
    print('\n🔴 Drift significatif — réentraînement du modèle à envisager')

print('\nRapport complet : reports/data_drift_report.html')